In [2]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
vocab = sorted(set(text))
vocab_size = len(vocab)
print(f"Dataset length: {len(text)} characters")
print(text[:250])
print(len(vocab))

Dataset length: 1115394 characters
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

65


In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
learning_rate = 1e-3
epochs = 3500
device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
str_to_int = {ch:i for i,ch in enumerate(vocab)}
int_to_str = {i:ch for i,ch in enumerate(vocab)}
encode = lambda s:[str_to_int[ch] for ch in s]
decode = lambda l:"".join([int_to_str[ID] for ID in l])
data=torch.tensor(encode(text),dtype=torch.long)

In [5]:
class GPTconfig:
    vocab_size: int
    block_size = 256
    batch_size = 64
    n_layer = 6
    n_head = 6
    n_embd = 384
    dropout = 0.2


In [6]:
class CasualMultiheadAttention(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.n_head = config.n_head
        self.head_dim = config.n_embd//config.n_head
        self.l1 = nn.Linear(config.n_embd,3*config.n_embd)
        self.register_buffer(
            "mask",
            torch.tril(torch.ones(config.block_size,config.block_size)
            .view(1,1,config.block_size,config.block_size))
        )
        self.l2 = nn.Linear(config.n_embd,config.n_embd)
    def forward(self,x):
        B,T,C = x.shape
        qkv = self.l1(x)
        q,k,v = qkv.split(C,dim=-1)
        q = q.view(B,T,self.n_head,self.head_dim).transpose(1,2)
        k = k.view(B,T,self.n_head,self.head_dim).transpose(1,2)
        v = v.view(B,T,self.n_head,self.head_dim).transpose(1,2)
        # calculate score
        score = q @ k.transpose(-2,-1)
        scaled_score = score/math.sqrt(self.head_dim)
        # apply mask
        masked_score = scaled_score.masked_fill(self.mask[:,:,:T,:T]==0,float('-inf'))
        # getting probabilities
        prob = F.softmax(masked_score,dim=-1)
        # Agregate values
        output = prob @ v
        output = output.transpose(1,2).contiguous().view(B,T,C)
        output = self.l2(output)
        return output

In [7]:
class MLP(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.inputLayer = nn.Linear(config.n_embd,4*config.n_embd)
        self.gelu = nn.GELU()
        self.hiddenLayer = nn.Linear(4*config.n_embd,config.n_embd)
        self.drop = nn.Dropout(config.dropout)
    def forward(self,x):
        x = self.inputLayer(x)
        x = self.gelu(x)
        x = self.hiddenLayer(x)
        x = self.drop(x)
        return x

In [8]:
class Block(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.norm1 = nn.LayerNorm(config.n_embd)
        self.attn = CasualMultiheadAttention(config)
        self.norm2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)
        self.drop = nn.Dropout(config.dropout)
    def forward(self,x):
        x = x + self.drop(self.attn(self.norm1(x)))
        x = x + self.mlp(self.norm2(x))
        return x

In [9]:
class GPTModel(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.config = config
        # input layer
        self.wte = nn.Embedding(config.vocab_size,config.n_embd)
        self.wpe = nn.Embedding(config.block_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
        # main layer
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        # output layer
        self.f_norm = nn.LayerNorm(config.n_embd)
        self.l_head = nn.Linear(config.n_embd, config.vocab_size)
        self.l_head.weight = self.wte.weight

    def forward(self,idx,targets=None):
        B,T = idx.shape
        device = idx.device
        tokenEmb = self.wte(idx)
        pos = self.wpe(torch.arange(T,dtype=torch.long,device=device))
        x = tokenEmb + pos
        x = self.drop(x)
        for block in self.blocks:
            x = block(x)
        x = self.f_norm(x)
        logits = self.l_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1,self.config.vocab_size),targets.view(-1))
        return logits,loss
    @torch.no_grad()
    def generate(self,idx,max_new_token,temperature=1.0,topk=None):
        for i in range(max_new_token):
            idx_crop = idx[:,-self.config.block_size:]
            logits,_ = self(idx_crop)
            # to make the probabilites a bit random or more concrete
            logits = logits[:,-1,:]/temperature

             # (Optional) TOP-K filtering
            if topk is not None:
                v,_ = torch.topk(logits,min(topk,logits.size(-1)))
                logits[logits<v[:,[-1]]] = -float('Inf')

            pobs = F.softmax(logits,dim=-1)
            index = torch.multinomial(pobs,num_samples=1)
            idx = torch.cat((idx,index),dim=-1)
        return idx


In [10]:
n = int(len(data) * 0.9)
train_data = data[:n]
val_data = data[n:]
# context and targets
def get_batch(split='train'):
    if split == 'train':
        data = train_data
    else:
        data = val_data
    rand = torch.randint(len(data)-config.block_size,(config.batch_size,))

    context = torch.stack([data[ i : i+config.block_size] for i in rand])
    targets = torch.stack([data[ i+1 : i+config.block_size+1] for i in rand])
    return context.to(device),targets.to(device)

In [12]:
config = GPTconfig()
config.vocab_size = vocab_size
model = GPTModel(config).to(device)

In [13]:
@torch.no_grad()
def estimate_loss():
    out={}
    model.eval()
    for split in ['train','val']:
        losses=torch.zeros(100)
        for k in range(100):
            X,Y=get_batch(split)
            logits,loss=model(X,Y)
            losses[k]=loss.item()
        out[split]=(losses.mean()).item()
    model.train()
    return out
# optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
def train():
    for epoch in range(epochs+1):
        context,target=get_batch('train')
        context = context.to(device)
        target = target.to(device)
        logits,loss=model(context,target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if epoch % 250 ==0:
            losses=estimate_loss()
            print(f"Epoch {epoch:02d}: train loss={losses['train']:.4f} and val loss={losses['val']:.4f}")


In [14]:
losses=estimate_loss()
print(f"train loss={losses['train']:.4f} and val loss={losses['val']:.4f}")

train loss=250.8766 and val loss=250.4708


In [16]:
context,target = get_batch('val')
generated_chars=decode(model.generate(context,500,0.5,20)[0].tolist())
print(generated_chars)

rl,
Valance of Venice gold in needlework,
Pewter and brass and all things that belong
To house or housekeeping: then, at my farm
I have a hundred milch-kine to the pail,
Sixscore fat oxen standing in my stalls,
And all things answerable to this portion.
Myyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyy


In [18]:
context,target = get_batch()

logits,loss = model(context,target)
print("logits shape:",logits.shape)
print("target shape:",target.shape)

train()


logits shape: torch.Size([64, 256, 65])
target shape: torch.Size([64, 256])
Epoch 00: train loss=99.8767 and val loss=100.9812
Epoch 250: train loss=2.8442 and val loss=2.8673
Epoch 500: train loss=2.6054 and val loss=2.6312
Epoch 750: train loss=2.5822 and val loss=2.6166
Epoch 1000: train loss=2.4562 and val loss=2.4857
Epoch 1250: train loss=2.2374 and val loss=2.2846
Epoch 1500: train loss=2.0873 and val loss=2.1686
Epoch 1750: train loss=1.9934 and val loss=2.1307
Epoch 2000: train loss=1.9023 and val loss=2.0562
Epoch 2250: train loss=1.8564 and val loss=2.0173
Epoch 2500: train loss=1.7530 and val loss=1.9441


In [19]:
context,target = get_batch('val')
generated_chars=decode(model.generate(context,500,0.5,20)[0].tolist())
print(generated_chars)

,--
And he shall be Vincentio of Pisa;
And make assurance here in Padua
Of greater sums than I have promised.
So shall you quietly enjoy your hope,
And marry sweet Bianca with consent.

LUCENTIO:
Were it not that my fellow-school-master
Doth watch Bianca's if those you.

KING EDWARD IV:
ISABEL:
And the see the sire me be in stand sounder ese shall stis stereate,
And be the be so she will he shall begive to here shall me peak of the their seest
Thim that of the is of the be dis and mese so him him.

SAUREN MARD:
When mothe stell blost me so ther such to your him thy the more my stand the
The withe sher she see premine he shereal I his do dom there she
To proft boy behou hem thereet so be say this her some:
The being he profore hen the sing the sle


In [21]:
context,target = get_batch()

logits,loss = model(context,target)
print("logits shape:",logits.shape)
print("target shape:",target.shape)
# print("initial loss:",loss.item())
train()

logits shape: torch.Size([64, 256, 65])
target shape: torch.Size([64, 256])
Epoch 00: train loss=1.7427 and val loss=1.9259
Epoch 250: train loss=1.6601 and val loss=1.8419
Epoch 500: train loss=1.6077 and val loss=1.8218


In [22]:
context,target = get_batch('val')
generated_chars=decode(model.generate(context,500,0.5,20)[0].tolist())
print(generated_chars)


GONZALO:
Had I plantation of this isle, my lord,--

ANTONIO:
He'ld sow't with nettle-seed.

SEBASTIAN:
Or docks, or mallows.

GONZALO:
And were the king on't, what would I do?

SEBASTIAN:
'Scape being drunk for want of wine.

GONZALO:
I' the commonwealth therefors, sir,
And of that him bashere tood, the pack to the part,
And of the of shall have man this forset.

CORDIOLANUS:
The shall the our the man this back'd by the batter
And that have the the saitore.
May have the against the shals that bears and again,
And and heart, and sportheir the stable.

CORIOLANUS:
And Mark!

MERCALUS:
Sinst for the be a says, take again the with the ward?

MENENIUS:
God that to hast than here the way when the made shall hear with.

CORIOLANUS:
And Seat the will th
